In [ ]:
import lib_dasilva2026
import numpy as np
import importlib
from mpl_toolkits.axes_grid1 import make_axes_locatable

importlib.reload(lib_dasilva2026)

In [ ]:
lib_dasilva2026.load_data?

In [ ]:
tracers_data = lib_dasilva2026.load_data(
    aci_file='data/myrun/aci/ts2_l2_aci_ipd_20251116_v1.0.0.cdf',
    ead_file='data/myrun/ead/ts2_def_ead_20251116_v0.10.1.cdf',
)

In [ ]:
lib_dasilva2026.plot_spect(tracers_data)

In [ ]:
Eic = lib_dasilva2026.find_Eic(tracers_data, smooth=True)

In [ ]:
def get_iflux_at_Eic(data, Eic):
    iflux_at_Eic = np.zeros_like(Eic)
    for i in range(data.spect.shape[0]):
        if np.isnan(Eic[i]):
            iflux_at_Eic[i] = np.nan
        else:
            iflux_at_Eic[i] = data.spect[i, np.searchsorted(data.energies, Eic[i])]
    return iflux_at_Eic

In [ ]:
MAX_SHEATH_ENERGY = 3.1e3
MIN_ION_VALID_ENERGY = 50
MIN_AVG_IFLUX_SHEATH = 10**6
MIN_IFLUX_AT_EIC = 10**6     


def test_detection(tracers_data, start_time, end_time, omni_data):
    data_subset = tracers_data.subset(start_time, end_time)
    Eic = lib_dasilva2026.find_Eic(data_subset, smooth=True, window_size=11)
    ch_i = tracers_data.energies.searchsorted(MIN_ION_VALID_ENERGY)
    ch_j = tracers_data.energies.searchsorted(MAX_SHEATH_ENERGY)
    iflux_avg_sheath = np.mean(data_subset.spect.T[ch_i:ch_j, :], axis=0)
    iflux_peak_sheath = np.max(data_subset.spect.T[ch_i:ch_j, :], axis=0)
    iflux_at_Eic = get_iflux_at_Eic(data_subset, Eic)
    
    iflux_avg_sheath_mask = iflux_avg_sheath > MIN_AVG_IFLUX_SHEATH
    iflux_at_Eic_mask = iflux_at_Eic > MIN_IFLUX_AT_EIC
    Eic_in_sheath_mask = Eic < MAX_SHEATH_ENERGY
    mask = iflux_avg_sheath_mask & iflux_at_Eic_mask & Eic_in_sheath_mask
    
    delta_t = [dt.total_seconds() for dt in np.diff(data_subset.time)]   
    D = np.diff(np.log10(Eic)) / delta_t
    D *= - np.sign(np.diff(data_subset.mlat))
    D[~mask[:-1]] = 0
    D[Eic[:-1] > MAX_SHEATH_ENERGY] = 0

    time_curr = start_time + (end_time - start_time) / 2
    time_curr_d2n = date2num(time_curr)
    Bx = np.interp(time_curr_d2n, omni_data['time_d2n'], omni_data['Bx'])
    By = np.interp(time_curr_d2n, omni_data['time_d2n'], omni_data['By'])
    Bz = np.interp(time_curr_d2n, omni_data['time_d2n'], omni_data['Bz'])
    
    fig, axes = plt.subplots(4, 1, figsize=(8, 8), sharex=True)

    fig.suptitle(
        f"Dispersion Event: {start_time.strftime('%Y-%m-%d')},  {start_time.strftime('%H:%M:%S')} - {end_time.strftime('%H:%M:%S')} UT"
        f'\nIMF = <{Bx:.1f}, {By:.1f}, {Bz:.1f}> nT'
    )
    
    _, im = lib_dasilva2026.plot_spect(data_subset, fig=fig, ax=axes[0])
    axes[0].plot(data_subset.time[:-1][D != 0], Eic[:-1][D != 0], 'b-', label='$E_{ic}$')
    axes[0].axhline(MAX_SHEATH_ENERGY, color='k', linestyle='dashed',
                    label='Maximum Energy]\nfor Sheath Origin')
    axes[0].set_title('Ion Spectrogram from ACI')
    
    axes[1].plot(data_subset.time, iflux_avg_sheath, color='C2')
    axes[1].set_title('Average Ion Flux in Energy Range for Magnetosheath Origin')
    axes[1].set_ylabel('Averaged Omni Flux')
    axes[1].axhline(MIN_AVG_IFLUX_SHEATH, color='k', linestyle='dashed', label='Threshold')

    axes[2].plot(data_subset.time, iflux_at_Eic, color='C3')
    axes[2].set_title('Ion Flux at $E_{ic}$')
    axes[2].set_ylabel('Omni Flux')
    axes[2].axhline(MIN_IFLUX_AT_EIC, color='k', linestyle='dashed', label='Threshold')

    label = r'D(T) : Scoring Function'
    label += '\nTotal Score: %.2f' % np.sum(D * delta_t)
    axes[3].fill_between(data_subset.time[:-1], 0, D, label=label)
    axes[3].set_ylabel('D(t)')
    for ax in axes:
        ax.legend()
        divider = make_axes_locatable(ax)
        cax = divider.append_axes('right', size='5%', pad=0.05)
        cb = fig.colorbar(im, cax=cax, orientation='vertical')
        cb.set_label(r'Summed Omni Flux')

In [ ]:
from matplotlib.dates import date2num

omni_data = lib_dasilva2026.get_omni(tracers_data)
omni_data['time_d2n'] = date2num(omni_data['time'])

In [ ]:
from datetime import datetime

start_time = datetime(2025, 11, 16, 21, 51)
end_time = datetime(2025, 11, 16, 21, 52)

test_detection(tracers_data, start_time, end_time, omni_data)